**Note**: the API Documentation appears to be out-of-date, rely on the source code instead

https://github.com/microsoft/agent-framework/blob/main/python/packages/core/agent_framework/_types.py

`Mapping`: a dict-like subclass that means the function promises NOT to alter the input data
`MutableMapping`: a dict-like subclass that means the function allows data alteration

# Calling Agents Directly

In [ ]:
from agent_framework import Agent, tool, FunctionInvocationContext
from agent_framework._types import AgentResponse, AgentResponseUpdate, ContinuationToken, StructuredResponseFormat
from pydantic import Field
from typing import Annotated, TypedDict, Literal, Any, Mapping, MutableMapping, Sequence
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from agent_framework.openai import OpenAIChatClient  # Responses API
load_dotenv()

True

In [11]:
# Tool without approval mode
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str | int, Field(description="The location to get the weather for.")],
    ctx: FunctionInvocationContext,
) -> str | int:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [ ]:
base_url = os.getenv("FOUNDRY_OPENAI_BASE_URL")
if not base_url:
    raise ValueError("FOUNDRY_OPENAI_BASE_URL environment variable is not set.")

model = os.getenv("FOUNDRY_MODEL")
if not model:
    raise ValueError("FOUNDRY_MODEL environment variable is not set.")

In [13]:
openai_client = OpenAIChatClient(
    model=model,
    base_url=base_url,
    credential=DefaultAzureCredential()
)

agent = Agent(
    client=openai_client,
    name="WeatherAgent",
    description="A helpful weather assistant that can answer questions about the weather.",
    instructions=(
        "You are a helpful weather assistant with access to a get_weather tool for answering questions about the weather. "
    ),
    tools=[get_weather],
    context_providers=[],
)

In [22]:
result = agent.run("What is the weather in Halifax?", stream=True)

response_set = set()

async for message in result:
    # AgentResponseUpdate
    response_set.add(type(message))

# AgentResponse
final = await result.get_final_response()
print(dir(final))
response_set.add(type(final))

print("\n".join([str(t) for t in response_set]))

['DEFAULT_EXCLUDE', 'INJECTABLE', '_SHALLOW_COPY_FIELDS', '__annotate_func__', '__annotations__', '__annotations_cache__', '__class__', '__class_getitem__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_get_type_identifier', '_response_format', '_value', '_value_parsed', 'additional_properties', 'agent_id', 'continuation_token', 'created_at', 'finish_reason', 'from_dict', 'from_json', 'from_update_generator', 'from_updates', 'messages', 'raw_representation', 'response_id', 'text', 'to_dict', 'to_json', 'usage_details', 'user_input_requests', 'value']
<class 'agent_framework._types.AgentResponse

## AgentResponse

Represents the response to an Agent run request.

Provides one or more response messages and metadata about the response. A typical response will contain a single message, but may contain multiple messages in scenarios involving function calls, RAG retrievals, or complex logic.

In [ ]:
class AgentResponseType(TypedDict):
    messages: Message | Sequence[Message] | None
    response_id: str | None
    agent_id: str | None
    created_at: CreatedAtT | None
    finish_reason: FinishReasonLiteral | FinishReason | None
    usage_details: UsageDetails | None
    value: ResponseModelT | None
    response_format: StructuredResponseFormat | None # pydantic basemodel or Mapping[str, Any]
    continuation_token: ContinuationToken | None # Used to poll for completion or resume a streaming response
    raw_representation: Any | None
    additional_properties: dict[str, Any] | None

## AgentResponseUpdate

Represents a single streaming response chunk from an Agent.

## Message

Represents a change message

In [ ]:
# LEFT-OFF: confirming content type
# LEFT-OFF: confirming workflow types (after this)
# LEFT-OFF: need to reconcile these changes with the changes on my local PC


RoleLiteral = Literal["system", "user", "assistant", "tool"]

class MessageType(TypedDict):
    role: RoleLiteral | str
    contents: Sequence[Content | str | Mapping[str, Any]] | None
    author_name: str | None
    message_id: str | None
    additional_properties: MutableMapping[str, Any] | None
    raw_representation: Any | None

### Deprecated

- `ChatAgent`
- `ChatMessage`
- `ChatMiddelware`
- `Chat*`

All `Chat` prefixes were from the original API and are now considered deprecated. They have been migrated to representations without the `Chat` prefix (e.g., `Agent`).

# Calling Agents through Workflow